In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%%RecordEvent
from pathlib import Path
#
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import time

colors = ["gold", "mediumturquoise", "darkorange", "lightgreen"]
kaggle_colors = ["#39c5ff", "#ffffff", "#f2f2f2", "#9ca3a4", "#3f484b"]
font = "Rubik"

In [ ]:
### cell 0 ###

benchmark_name = "beautiful-kaggle-2022-analysis"
df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "kaggle-survey-2022" / "kaggle_survey_2022_responses.csv"
)
factor = 1
df = pd.concat([df] * factor, ignore_index=True)
df.info()

In [ ]:
### cell 1 ###

df = df[1:]

In [ ]:
### cell 2 ###

df["Duration (in seconds)"] = pd.to_numeric(
    df["Duration (in seconds)"], errors="coerce"
)

In [ ]:
### cell 3 ###

# Optimized: use a single vectorized replace to reduce GPU calls from 3 to 1
df["Q3"] = df["Q3"].replace(["Nonbinary", "Prefer not to say", "Prefer to self-describe"], "Others")

In [ ]:
### cell 4 ###

gender_count = df.groupby(["Q3"]).size().reset_index().rename(columns={0: "count"})

In [ ]:
### cell 5 ###

duration_count = (
    df.groupby("Duration (in seconds)")
      .size()
      .reset_index(name="count")
      .head(1000)
)

In [ ]:
### cell 6 ###

age_count = df.groupby("Q2").size().reset_index(name="count")

In [ ]:
### cell 7 ###

country_count = (
    df.groupby('Q4')
      .size()
      .to_frame('count')
      .sort_values('count', ascending=False)
      .head(10)
      .reset_index()
)

In [ ]:
### cell 8 ###

country_df = df['Q4'].value_counts().reset_index(name='count').rename(columns={'index': 'Q4'})

In [ ]:
### cell 9 ###

student_count = df.groupby("Q5", sort=False).size().reset_index(name="count")

In [ ]:
### cell 10 ###

degree_count = (
    df.groupby("Q8")
      .size()
      .reset_index(name="count")
)
# replace in one vectorized GPU call instead of boolean indexing
degree_count["Q8"] = degree_count["Q8"].replace(
    "Some college/university study without earning a bachelor’s degree",
    "College Without Bachelors"
)

In [ ]:
### cell 11 ###

experience_count = df['Q16'].value_counts().reset_index()
experience_count.columns = ['Q16', 'count']

In [ ]:
### cell 12 ###

annual_income_df = df.groupby(["Q29"]).size().reset_index().rename(columns={0: "count"})

In [ ]:
### cell 13 ###

gender_duration_count = (
    df.groupby(["Duration (in seconds)", "Q3"])
    .size()
    .reset_index()
    .rename(columns={0: "count"})
)
gender_duration_count = gender_duration_count[:3000]

In [ ]:
### cell 14 ###

country_gender = df.groupby(["Q4", "Q3"])['Q4']\
    .count()\
    .reset_index(name="count")

In [ ]:
### cell 15 ###

age_duration_df = (
    df[['Q2', 'Duration (in seconds)']]                               # only pull in the two columns we need
      .loc[lambda d: d['Duration (in seconds)'] < 2000]                # filter on GPU without an extra __getitem__ pass
      .groupby('Q2', as_index=False)                                   # as_index=False eliminates the need for reset_index()
      .mean()                                                         # vectorized mean on GPU
      .round(1)                                                       # vectorized round on GPU
)

In [ ]:
### cell 16 ###

education_gender = (
    df.groupby(["Q8", "Q3"]).size().reset_index().rename(columns={0: "count"})
)

In [ ]:
### cell 17 ###

## Not Including India bcz it acts as an outlier here
top_9_countries = [
    "United States of America",
    "Other",
    "Brazil",
    "Nigeria",
    "Pakistan",
    "Japan",
    "China",
    "Egypt",
    "Mexico",
]

top_9_countries_df = df[df["Q4"].isin(top_9_countries)]

In [ ]:
### cell 18 ###

gender_student = (
    df.groupby(["Q3", "Q5"], as_index=False)
      .size()
      .rename(columns={"size": "count"})
)